In [56]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-18 20:30:32--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  3.86MB/s    in 0.3s    

2026-08-18 20:30:32 (3.86 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [57]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [58]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [59]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [60]:
chars = sorted(list(set(text)))
vocab_size  = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [61]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hi there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hi there


In [62]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [63]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [64]:
block_size= 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [65]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    cntx = x[:t+1]
    target = y[t]
    print(f"when input is {cntx} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [66]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('---')
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"whem input is {context.tolist()} the target: {target}")

inputs
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
whem input is [24] the target: 43
whem input is [24, 43] the target: 58
whem input is [24, 43, 58] the target: 5
whem input is [24, 43, 58, 5] the target: 57
whem input is [24, 43, 58, 5, 57] the target: 1
whem input is [24, 43, 58, 5, 57, 1] the target: 46
whem input is [24, 43, 58, 5, 57, 1, 46] the target: 43
whem input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
whem input is [44] the target: 53
whem input is [44, 53] the target: 56
whem input is [44, 53, 56] the target: 1
whem input is [44, 53, 56, 1] the target: 58
whem input is [44, 53, 56, 1, 58] the target: 46
whem input is [44, 53, 5

In [67]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1,), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [68]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [69]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


In [70]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecor


In [71]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 32])

In [72]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

In [73]:
xbow[0]

tensor([[ 1.2794, -0.7354, -0.2471,  0.9398,  2.0026, -1.3095,  0.3397,  0.2184,
         -1.9841,  0.3335,  0.4053,  1.8873,  1.2212, -1.2814,  0.7391,  0.6104,
         -1.9445,  0.1556, -0.3837, -1.3987,  0.6884,  1.2369, -0.5307,  0.3360,
          0.4970, -0.2665,  0.4055,  1.2194, -0.3673, -0.4904,  1.2458,  0.0990],
        [ 0.3030, -0.2433, -0.7831,  0.5083,  1.8772, -1.4463,  0.0874,  0.7034,
         -1.5372,  0.6351,  0.3666,  0.6393,  0.0911, -1.0762,  0.3247,  0.3711,
         -0.1536,  0.4245, -0.2753, -0.9046,  0.8555,  1.4164, -0.4916,  0.3386,
          0.2965, -0.1772,  0.2687,  1.0845,  0.5134, -0.5976,  0.9727,  0.0245],
        [ 0.0687, -0.2307, -0.4002,  0.0680,  1.0530, -0.7030,  0.1947,  0.3581,
         -0.7489,  0.8397,  0.2524,  0.5677,  0.2461, -0.7114, -0.1352,  0.0074,
         -0.1392,  0.0904, -0.3313, -1.0043,  0.4916,  0.8380, -0.0400,  0.1793,
          0.0864,  0.2403,  0.4470,  1.0109,  0.8565,  0.0223,  0.2424,  0.1136],
        [-0.2002, -0.6476

In [74]:
wei = torch.tril(torch.ones(T, T))
wei = wei/wei.sum(1, keepdim=True)
xbow2 = wei @ x
xbow2

tensor([[[ 1.2794, -0.7354, -0.2471,  ..., -0.4904,  1.2458,  0.0990],
         [ 0.3030, -0.2433, -0.7831,  ..., -0.5976,  0.9727,  0.0245],
         [ 0.0687, -0.2307, -0.4002,  ...,  0.0223,  0.2424,  0.1136],
         ...,
         [ 0.0990, -0.5090,  0.0668,  ..., -0.5093,  0.4030, -0.1157],
         [ 0.0089, -0.2780,  0.0854,  ..., -0.3191,  0.5163, -0.0188],
         [ 0.1493, -0.6081,  0.0942,  ..., -0.2632,  0.4732, -0.0391]],

        [[-0.5766,  0.1689,  0.4061,  ...,  0.7307, -1.4253,  2.0070],
         [ 0.6086, -0.0773,  0.0208,  ...,  0.7028,  0.1778,  1.5235],
         [ 0.7362, -0.1014, -0.2416,  ...,  0.9929,  0.5543,  0.5435],
         ...,
         [ 0.4455, -0.8160, -0.2110,  ...,  0.6389,  0.0714,  0.2867],
         [ 0.2798, -0.7018, -0.0873,  ...,  0.2632, -0.1075,  0.3128],
         [ 0.4026, -0.6018, -0.1666,  ...,  0.4204, -0.0587,  0.1205]],

        [[-0.3981,  0.0500,  0.6014,  ..., -1.4026,  0.6643,  1.9296],
         [-0.0900,  1.0072, -0.2984,  ..., -0

In [75]:
tril = torch.tril(torch.ones(T, T))
wei = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
xbow3

tensor([[[ 1.2794, -0.7354, -0.2471,  ..., -0.4904,  1.2458,  0.0990],
         [ 0.3030, -0.2433, -0.7831,  ..., -0.5976,  0.9727,  0.0245],
         [ 0.0687, -0.2307, -0.4002,  ...,  0.0223,  0.2424,  0.1136],
         ...,
         [ 0.0990, -0.5090,  0.0668,  ..., -0.5093,  0.4030, -0.1157],
         [ 0.0089, -0.2780,  0.0854,  ..., -0.3191,  0.5163, -0.0188],
         [ 0.1493, -0.6081,  0.0942,  ..., -0.2632,  0.4732, -0.0391]],

        [[-0.5766,  0.1689,  0.4061,  ...,  0.7307, -1.4253,  2.0070],
         [ 0.6086, -0.0773,  0.0208,  ...,  0.7028,  0.1778,  1.5235],
         [ 0.7362, -0.1014, -0.2416,  ...,  0.9929,  0.5543,  0.5435],
         ...,
         [ 0.4455, -0.8160, -0.2110,  ...,  0.6389,  0.0714,  0.2867],
         [ 0.2798, -0.7018, -0.0873,  ...,  0.2632, -0.1075,  0.3128],
         [ 0.4026, -0.6018, -0.1666,  ...,  0.4204, -0.0587,  0.1205]],

        [[-0.3981,  0.0500,  0.6014,  ..., -1.4026,  0.6643,  1.9296],
         [-0.0900,  1.0072, -0.2984,  ..., -0

In [76]:
B, T, C = 4, 8, 32
x= torch.randn(B, T, C)

head_size =16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5
print(wei.var())
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=1)
v = value(x)
out = wei@v
wei[0]

tensor(0.0929, grad_fn=<VarBackward0>)


tensor([[0.2331, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1219, 0.1276, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1566, 0.1382, 0.2063, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1603, 0.1300, 0.2031, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1297, 0.1464, 0.1354, 0.1960, 0.3155, 0.0000, 0.0000, 0.0000],
        [0.0875, 0.1573, 0.1305, 0.1709, 0.2367, 0.2557, 0.0000, 0.0000],
        [0.0471, 0.1101, 0.2910, 0.2243, 0.1800, 0.4105, 0.4644, 0.0000],
        [0.0882, 0.1601, 0.1067, 0.2058, 0.2678, 0.3338, 0.5356, 1.0000]],
       grad_fn=<SelectBackward0>)